In [ ]:
import tiktoken
import numpy as np

with open("/home/harsha/Projects/Astra/Optional/sample_dataset/canary_dataset.txt","r",encoding="utf-8") as f:
    content = f.read()

enc = tiktoken.get_encoding("gpt2")
tokens = enc.encode(content)

n = len(tokens)
split_idx = int(n*0.9)

train_tokens = tokens[:split_idx]
val_tokens = tokens[split_idx:]

train_array = np.array(train_tokens,dtype=np.uint16)
val_array  =  np.array(val_tokens,dtype=np.uint16)

train_array.tofile("train.bin")
val_array.tofile("val.bin")

print(f"Saved train.bin with {len(train_array)} tokens.")
print(f"Saved val.bin with {len(val_array)} tokens.")

# Tokenization with Tiktoken & Binary Saving

We import the `tiktoken` module and use the GPT-2 BPE (Byte-Pair Encoding) tokenizer (`gpt2`).

We encode the raw dataset text into integer token IDs, split the tokens into training (90%) and validation (10%) sets, and convert the Python lists into NumPy `uint16` arrays before saving them as binary `.bin` files on disk.


---

### Why Convert Python Lists to NumPy `uint16` & `.bin` Files?

There are two huge, practical reasons why standard Python lists break down for large datasets and why NumPy + binary files are mandatory:


### 1. Memory Bloat: 2 Bytes vs 28 Bytes Per Token

In pure Python, an integer is **not** just a raw number in memory. It is a heavy CPython heap object containing:
- Reference count (`ob_refcnt`, 8 bytes)
- Type pointer (`ob_type`, 8 bytes)
- Size metadata (8 bytes)
- Actual digit value (4-8 bytes)
- Plus the 8-byte pointer inside the Python `list` pointing to that object.

**Total cost in Python:** Around **28 to 36 bytes** per single integer.
If you have 100 million tokens in a Python list, that takes over **3.2 GB of RAM** just to sit in memory doing nothing.

**NumPy `uint16`:**
- Has zero object wrappers.
- Packs raw binary bytes side-by-side in contiguous memory.
- Because the GPT-2 vocabulary size is 50,257, every token ID fits cleanly in 16 bits (**exactly 2 bytes**).
- 100 million tokens in NumPy = **200 MB flat**. That is a 16x memory drop.


### 2. Binary Files (`.bin`) and `np.memmap`: Zero-RAM Dataset Streaming

If you save your dataset as JSON, CSV, or text:
1. Every time training starts, Python has to parse text, create millions of integer objects, and fill up gigabytes of RAM.
2. If your dataset grows to 10 GB or 50 GB, your machine will throw an `OutOfMemoryError` before training even begins.

When you save with `tofile("train.bin")`, NumPy dumps the contiguous byte stream straight to disk with no formatting or header overhead.

This unlocks **`np.memmap`** for training:
- `np.memmap` does **not** load the file into RAM.
- It tells the Linux OS kernel: *"Map this file directly into the virtual memory address space."*
- When the data loader requests a batch of 64 tokens (`data[idx : idx + 64]`), the OS page cache pulls *only those specific 128 bytes* directly off the SSD into your CPU cache.
- The rest of the file stays on disk. You can stream a 100 GB dataset on a machine with only 4 GB of RAM, and loading each batch takes microseconds.

Whenever you need it later, you don't even have to "read" or "load" the whole file. You just point `np.memmap("train.bin", dtype=np.uint16, mode="r")` at the file path and start slicing immediately.
